# DSI: emulate, validate, then condition

_History matching in seconds, instead of days._


Every notebook so far has paid the model's full price. Building it (["build the model"](../part1_01_build_model/dizon_build_model.ipynb)), setting up the parameters (["PstFrom setup"](../part1_02_pstfrom_setup/dizon_pstfrom_setup.ipynb)), choosing the weights and the synthetic truth (["observations, weights and truth"](../part1_03_obs_weights_and_truth/dizon_obs_weights_and_truth.ipynb)), and running the prior Monte Carlo (["prior Monte Carlo"](../part1_04_prior_mc/dizon_prior_mc.ipynb)) — all of it rests on running DIZON, and DIZON costs ~6 min per run. The prior ensemble alone was 201 of those runs.

That is the wall this whole curriculum is built against. A history match with PESTPP-IES needs several iterations of a few hundred runs each; for a ~6 min model that is days of compute for a single forecast. We cannot afford to do that every time the question changes.


**Data Space Inversion (DSI)** is the way around the wall. Instead of history-matching the model, we history-match a cheap statistical *emulator* built from runs we have already paid for. The prior Monte Carlo gave us 201 realisations of the model's outputs; DSI learns the joint behaviour of those outputs directly in observation space, and can then be conditioned on data in seconds. No new DIZON runs.

The discipline of this notebook is one rule, stated up front and obeyed every time: **never trust an emulator you have not tested.** So the order is

1. finalise the synthetic truth and hold it out of training;
2. train DSI and **check its fidelity first** on realisations it never saw;
3. only then condition it with PESTPP-IES;
4. read off the payoff — the prior vs posterior forecast distribution, and the compute bill.


### Admin

We need `pyemu` and `flopy` from the vendored `dependencies/` tree (so the DSI emulator and the corrected runstor forward-run code are the ones this curriculum was written against), plus `herebedragons` for the binary helper. Everything is run from inside this notebook's own directory.


In [ ]:
import os
import shutil
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil

import sys
import pyemu
import flopy
assert "dependencies" in flopy.__file__
assert "dependencies" in pyemu.__file__
sys.path.insert(0, "..")
import herebedragons as hbd

from pyemu.emulators import DSI

**Prerequisite check.** This notebook consumes two upstream products, and treats them differently *by design*:

- the **prior Monte Carlo ensemble** from ["prior Monte Carlo"](../part1_04_prior_mc/dizon_prior_mc.ipynb) is a hard prerequisite — there is nothing to emulate without it, so the cell below resolves the prior MC source in the canonical order — the tracked, thinned `prebaked/prior_mc_obs_ensemble.jcb` artifact first, then a full prior MC you ran into the repo-root `master_priormc` — and **raises** with a pointer to that notebook if neither is found. The prior MC results are a tracked, thinned `prebaked/` artifact — running 201 DIZON realisations took roughly 1.6 h across 10 workers on a MacBook (~6 min/run in serial), so we load them rather than re-running.
- the **synthetic truth** from ["observations, weights and truth"](../part1_03_obs_weights_and_truth/dizon_obs_weights_and_truth.ipynb) is *not* a file we read back. As that notebook says explicitly, it **stages the selection criterion and defers the actual pick to here**, because the criterion needs the prior forecast distribution, which only exists once `part1_04` has run. So we deliberately re-apply the part1_03 criterion below rather than load a truth file — keeping the sequence strictly ordered, with no notebook depending on outputs of a later one.


In [ ]:
# resolve the prior Monte Carlo source, with the canonical preference order:
#   1. the tracked, thinned prebaked obs ensemble (../../prebaked/prior_mc_obs_ensemble.jcb)
#   2. a full prior MC you ran yourself into the repo-root master dir (../../master_priormc)
# part1_04 uses the same order; we need a control file + tdis here, so we read those from
# whichever master directory carries them (the prebaked .jcb ships its companion pst).
prebaked_oe = Path("..") / ".." / "prebaked" / "prior_mc_obs_ensemble.jcb"
pmc_d = Path("..") / ".." / "master_priormc"

if prebaked_oe.exists() and (prebaked_oe.parent / "pest.pst").exists():
    # the prebaked artifact and its companion control file
    pmc_d = prebaked_oe.parent
    pst = pyemu.Pst(str(pmc_d / "pest.pst"))
    prior_oe_path = prebaked_oe   # named artifact; loaded with from_binary below
    print(f"loaded prebaked prior MC control file from {pmc_d}")
elif (pmc_d / "pest.pst").exists():
    # fall back to the full prior MC master dir produced during development
    pst = pyemu.Pst(str(pmc_d / "pest.pst"))
    prior_oe_path = None   # a live master dir; use the pst.ies.obsen accessor below
    print(f"loaded prior MC control file from {pmc_d}")
else:
    raise Exception(
        "you need to run the '../part1_04_prior_mc/dizon_prior_mc.ipynb' notebook "
        "(prior Monte Carlo results not found at the prebaked artifact "
        f"'{prebaked_oe}' or the master dir '{pmc_d}')")

In [ ]:
# The synthetic truth is picked here, not loaded. By design, part1_03 stages the
# selection *criterion* (a realisation from the upper quartile of the prior forecast
# distribution) but defers the actual pick to this notebook, because the criterion
# needs the prior forecast distribution that only exists after part1_04 has run. So
# we re-apply that criterion below rather than read a truth file. There is
# intentionally no synthetic_truth.csv to load.
print("synthetic truth: applying the part1_03 selection criterion to the prior "
      "ensemble below (deliberate; part1_03 stages the criterion, part1_05 makes "
      "the pick)")


## 1. Finalise the synthetic truth

The canonical forecast is **peak SO₄ at the supply well (`wellopt`) during the supply period (day 308–728)** — the maximum over *all* supply-well screens (`welopt-ly1`, `welopt-ly3`, `welopt-ly5`) and supply-period times. Treatment cost scales with that concentration, so the operator designs treatment capacity to its P95; the payoff we track is the whole forecast distribution, prior vs posterior. In ["observations, weights and truth"](../part1_03_obs_weights_and_truth/dizon_obs_weights_and_truth.ipynb) we staged the selection criterion: pick a prior realisation from the **upper quartile** of the prior forecast distribution (peak SO₄ ≈ 87–95 mg/L), so the prior median under-predicts the truth and conditioning has something visible to correct toward. Here we apply that criterion to the prior ensemble on disk, finalise the truth, show it, name it, and — critically — **drop it from the training set** before any emulator sees it.


First, the model's stress-period boundaries. The prior MC stored a dense concentration time series; for DSI we subsample to the stress-period boundary times, which keeps the breakthrough shape while keeping the emulator's output dimension manageable:


In [ ]:
# stress-period boundary times: prefer the flow model's tdis if part1_01's
# workspace is on disk; otherwise derive from the thinned control file's obs
# metadata (the curated set keeps conditioning obs at exactly these times)
gwf_ws = Path("..", "part1_01_build_model", "model")
if (gwf_ws / "mfsim.nam").exists():
    sim = flopy.mf6.MFSimulation.load(sim_ws=str(gwf_ws), load_only=["tdis"],
                                      verbosity_level=0)
    perioddata = pd.DataFrame(sim.tdis.perioddata.get_data())
    modeltimes = perioddata.perlen.cumsum().values
else:
    od = pst.observation_data
    cond = od.loc[(od.weight > 0) & (od.time.astype(float) > 0)]
    modeltimes = np.array(sorted(cond.time.astype(float).unique()))
print(f"{len(modeltimes)} stress-period boundary times, day "
      f"{modeltimes.min():.0f} to {modeltimes.max():.0f}")

Pull the prior observation ensemble and keep only the concentration time-series observations at those boundary times. These are the columns DSI will learn:


In [ ]:
# re-parse the obsname tokens so oname/obsid/variable/time reflect the immutable
# obsname (part1_02 overwrites the oname column for convenience; this restores it,
# the same call part1_04 makes before selecting concentration obs)
pst.try_parse_name_metadata()
obs = pst.observation_data.copy()
obs["time"] = pd.to_numeric(obs["time"], errors="coerce")

# the time-series concentration obs live in the 'conc' observation type; select by
# oname (parsed from the immutable obsname token) not obgnme, which part1_02/part1_03
# retag to "forecast" and "<obsid>:<variable>" groups
conc = obs.loc[obs.oname == "conc"].copy()
keepobs = conc.loc[conc.time.isin(modeltimes)].obsnme.tolist()
print(f"keeping {len(keepobs)} concentration observations for DSI training")

In [ ]:
# the prior obs ensemble (iteration 0 of the prior MC = the prior)
if prior_oe_path is not None:
    # the tracked prebaked artifact is a named .jcb, not a pestpp output file
    oe = pyemu.ObservationEnsemble.from_binary(pst=pst, filename=str(prior_oe_path))
else:
    oe = pst.ies.obsen
oe = oe._df if hasattr(oe, "_df") else pd.DataFrame(oe)
data = oe.loc[:, keepobs].copy()
print(f"prior obs ensemble: {data.shape[0]} realisations x {data.shape[1]} obs")

Now the forecast. Concentrations in this model are carried in **mol/L**, so we convert SO₄ to mg/L (molar mass 96.06 g/mol). Peak SO₄ is the maximum over the supply window *and* over all three supply-well screens:


In [ ]:
SO4_MW = 96.06  # g/mol

# the supply-well forecast spans all three screens (welopt-ly1/ly3/ly5)
FORECAST_SCREENS = ["welopt-ly1", "welopt-ly3", "welopt-ly5"]

def peak_so4_mgl(ensemble, screens=FORECAST_SCREENS):
    """Peak supply-period SO4 (mg/L), max over all supply-well screens, per realisation."""
    fc = conc.loc[conc.obsid.isin(screens) & (conc.variable == "so4")
                  & (conc.time >= 308) & (conc.time <= 728)]
    cols = [c for c in fc.obsnme if c in ensemble.columns]
    return ensemble.loc[:, cols].max(axis=1) * 1000.0 * SO4_MW  # mol/L -> mg/L

prior_peak = peak_so4_mgl(oe)
print(f"prior peak SO4 at supply well (mg/L): "
      f"min={prior_peak.min():.1f}  median={prior_peak.median():.1f}  "
      f"P95={prior_peak.quantile(0.95):.1f}  max={prior_peak.max():.1f}")


The part1_03 criterion picks a realisation from the **upper quartile** of the prior forecast distribution — concretely, the one whose peak SO₄ sits nearest the median of that upper quartile, so the truth lands in the 87–95 mg/L band above the prior median. We apply it, name the truth, and look at it:


In [ ]:
# part1_03 criterion, applied verbatim: the realisation whose peak supply-well SO4
# (max over all supply screens and supply-period times) is nearest the prior 87.5th
# percentile -- the middle of the upper quartile. Deterministic, lands in the
# 87-95 mg/L band, and the prior median under-predicts it.
TRUTH_PCTILE = 0.875   # same constant as part1_03
target = prior_peak.quantile(TRUTH_PCTILE)
truth_real = (prior_peak - target).abs().idxmin()
print(f"target (prior {TRUTH_PCTILE:.0%} percentile): {target:.1f} mg/L")
print(f"synthetic truth = realisation '{truth_real}', "
      f"peak SO4 = {prior_peak.loc[truth_real]:.1f} mg/L "
      f"(prior median {prior_peak.median():.1f} mg/L under-predicts it)")

# the truth as a single row of observation values (the DSI training columns)
truth = data.loc[[truth_real]].copy()
# tiny pre-breakthrough values are numerical zero
truth[truth.abs() < 1e-8] = 0.0

In [ ]:
# show the truth's supply-well SO4 breakthrough (central screen welopt-ly3, for display)
fc = conc.loc[(conc.obsid == "welopt-ly3") & (conc.variable == "so4")
              & (conc.time.isin(modeltimes))].sort_values("time")
fc_cols = [c for c in fc.obsnme if c in truth.columns]
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(fc.set_index("obsnme").loc[fc_cols, "time"].values,
        truth.loc[truth_real, fc_cols].values * 1000.0 * SO4_MW,
        "-o", ms=3, color="fuchsia", label="synthetic truth")
ax.axvspan(308, 728, color="orange", alpha=0.1, label="supply period")
ax.set_xlabel("time (d)")
ax.set_ylabel("SO$_4$ (mg/L)")
ax.set_title("supply well (welopt-ly3): the truth we are trying to recover",
             loc="left", fontsize="small")
ax.legend(fontsize="small")
fig.tight_layout()


Now **drop the truth from the data**. The emulator must never see it: the whole exercise is to recover a withheld reality from history-period data alone. We also hold back a handful of further realisations as a *validation set* — these are realisations the emulator will not train on, so we can check its predictions against the full model's actual outputs (the fidelity check, step 2):


In [ ]:
# drop the truth
data = data.drop(index=truth.index)

# hold out a validation set for the fidelity check
rng = np.random.default_rng(0)
val_reals = rng.choice(data.index.values,
                       size=min(20, data.shape[0] // 5),
                       replace=False)
validation = data.loc[val_reals].copy()
train = data.drop(index=val_reals).copy()
print(f"training realisations:   {train.shape[0]}")
print(f"validation realisations: {validation.shape[0]} (held out, never trained on)")
print(f"synthetic truth:         1 (held out)")

## 2. Fidelity check first

Before conditioning on anything, we train DSI and ask the only question that matters for an emulator: **can it reproduce model outputs it has never seen?** We have the perfect test set sitting idle — the validation realisations the prior MC already ran. We feed each one's *true* outputs through the emulator's encode/decode round-trip and compare against the full model's actual values. This costs nothing: no new DIZON runs.


### The naive fit (and why it fails)

DSI works in a transformed observation space. The instinct is to standardise the data (subtract the mean, divide by the standard deviation) and run the SVD on that. Let us do exactly that first — `standard_scaler` — and see what the emulator predicts for the held-out realisations:


In [ ]:
transforms = [{"type": "standard_scaler"}]
dsi_naive = DSI(data=train, transforms=transforms, energy_threshold=0.975)
dsi_naive.fit()
print(f"latent dimension (retained components): {dsi_naive.latent_dim}")

To reconstruct a held-out realisation we project its (transformed) outputs onto the latent space and decode. `predict` takes latent coordinates, so we encode the validation realisations into latent space first, then decode them back:


In [ ]:
def encode(dsi, df):
    """Project realisations (obs-space rows) onto DSI latent coordinates.

    Numerical care is load-bearing here. A naive least-squares projection of a
    vector that lies (even slightly) outside the training subspace can return
    latents with enormous norms, and the normal-score back-transform then
    extrapolates them quadratically into absurdity (we saw concentrations of
    1e16 mol/L before adding the two guards below). So:
    1. clip each transformed value into its training range -- outside it the
       normal-score transform is extrapolating, not interpolating;
    2. project with a rank-revealing pseudo-inverse rather than raw lstsq.
    """
    xt = dsi.transformer_pipeline.transform(df.copy())
    xt = xt.loc[:, dsi.ovals.index]
    # guard 1: stay inside the training range of the transformed space
    lo = dsi.data_transformed[dsi.ovals.index].min(axis=0)
    hi = dsi.data_transformed[dsi.ovals.index].max(axis=0)
    xt = xt.clip(lower=lo, upper=hi, axis=1)
    dev = xt.values - dsi.ovals.values[np.newaxis, :]
    # guard 2: stable projection
    pinv = np.linalg.pinv(dsi.pmat, rcond=1e-8)
    pvals = pinv @ dev.T
    return pd.DataFrame(pvals.T, index=df.index,
                        columns=[f"p_{i}" for i in range(dsi.pmat.shape[1])])


# encode + decode the held-out realisations with the naive emulator
val_latent_naive = encode(dsi_naive, validation)
val_pred_naive = dsi_naive.predict(val_latent_naive)

Now the tell-tale failure. Concentrations are physical: they cannot be negative. A standard-scaler fit reconstructs in an unbounded linear space, so the decoded predictions routinely go below zero — especially for the long pre-breakthrough stretches where the true concentration is exactly zero:


In [ ]:
neg_frac = (val_pred_naive.values < 0).mean()
min_pred = val_pred_naive.values.min()
print(f"fraction of predicted values that are NEGATIVE: {neg_frac:.1%}")
print(f"most negative predicted concentration (mol/L): {min_pred:.4e}")
print("\nNon-physical. An emulator that invents negative concentrations cannot "
      "be trusted to condition the forecast.")

In [ ]:
# look at it on one validation realisation's supply-well SO4 breakthrough
vr = validation.index[0]
fig, ax = plt.subplots(figsize=(7, 3))
t = fc.set_index("obsnme").loc[fc_cols, "time"].values
ax.plot(t, validation.loc[vr, fc_cols].values, "-o", ms=3, color="k",
        label="full model (truth for this real)")
ax.plot(t, val_pred_naive.loc[vr, fc_cols].values, "-s", ms=3, color="crimson",
        label="DSI (standard_scaler)")
ax.axhline(0.0, color="grey", lw=0.8)
ax.set_xlabel("time (d)")
ax.set_ylabel("SO$_4$ (mol/L)")
ax.set_title(f"held-out realisation {vr}: standard_scaler predicts negative SO$_4$",
             loc="left", fontsize="small")
ax.legend(fontsize="small")
fig.tight_layout()

### The fix: a normal-score transform

The cure is to give DSI a transformed space where the SVD's Gaussian assumptions hold *and* the back-transform stays inside the physical range. The **normal-score transform** maps each observation's marginal to a standard normal using the empirical distribution from the training ensemble; the inverse maps strictly back within the range the training data spanned. With quadratic tail extrapolation the inverse is exact and monotone, so a reconstructed value cannot fall below the smallest training value for that observation — and for the pre-breakthrough zeros, that floor is zero.

(`log10` is the other common choice for concentrations, and the vendored transformer handles zeros via a learned shift. We prefer `normal_score` here because the breakthrough series mixes hard zeros with a skewed positive tail, which the empirical normal score handles without a single global shift.)


In [ ]:
transforms = [{"type": "normal_score", "quadratic_extrapolation": True}]
dsi = DSI(data=train, transforms=transforms, energy_threshold=0.99)
dsi.fit()
# energy_threshold matters: at 0.999 the latent space keeps ~170 components for
# ~190 training realisations -- nearly full rank, so conditioning can "fit" the
# history by composing noise-level components that carry no real information
# about the forecast (we learned this the hard way: the posterior detached from
# the truth entirely). 0.975 keeps the dominant, genuinely-correlated structure.
print(f"latent dimension (retained components): {dsi.latent_dim}")

In [ ]:
val_latent = encode(dsi, validation)
val_pred = dsi.predict(val_latent)

neg_frac = (val_pred.values < -1e-9).mean()
print(f"fraction of predicted values that are meaningfully NEGATIVE: {neg_frac:.2%}")
print(f"most negative predicted concentration (mol/L): {val_pred.values.min():.2e}")

# the fidelity check is not just a plot -- it is a MEASUREMENT of the emulator's
# error, per observation, on realisations it never saw. It is the evidence that
# earns the emulator the right to be conditioned. We report it as a diagnostic;
# we do NOT fold it into the noise budget below -- part1_03's measurement noise
# is used as-is.
emu_rmse = ((val_pred[validation.columns] - validation) ** 2).mean(axis=0) ** 0.5
print(f"emulator per-obs RMSE: median {emu_rmse.median():.3g}, "
      f"max {emu_rmse.max():.3g} (model units)")

Essentially no negatives — the handful that remain are tiny tail-extrapolation artefacts at the floor, orders of magnitude smaller than the -27 mol/L excursions the standard-scaler produced, and they vanish if we clip at zero. Now the real fidelity test: does the emulator reproduce the **forecast** for realisations it never saw? We compare DSI's predicted peak SO₄ against the full model's actual peak SO₄ across the validation set. Points on the 1:1 line mean the emulator is faithful where it counts:


In [ ]:
true_peak = peak_so4_mgl(validation)
pred_peak = peak_so4_mgl(val_pred)
common = true_peak.index.intersection(pred_peak.index)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
lim = [0, max(true_peak.max(), pred_peak.max()) * 1.1]
ax.plot(lim, lim, "k--", lw=1, label="1:1")
ax.scatter(true_peak.loc[common], pred_peak.loc[common], s=25,
           color="#1f77b4", alpha=0.8)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("full model peak SO$_4$ (mg/L)")
ax.set_ylabel("DSI peak SO$_4$ (mg/L)")
ax.set_title("fidelity check: held-out realisations", loc="left", fontsize="small")
ax.legend(fontsize="small")
fig.tight_layout()

The series principle, restated now that we have seen both halves of it: **never trust an emulator you have not tested.** The naive fit looked plausible until we asked it for a held-out prediction; the normal-score fit earns the right to be conditioned. With the fidelity check passed, we proceed.


## 3. Condition the emulator with PESTPP-IES

Conditioning the emulator means running PESTPP-IES against it instead of against DIZON. `DSI.prepare_pestpp` writes a complete PEST++ interface into a template directory: a control file (`dsi.pst`), a pickled emulator (`dsi.pickle`), and a forward-run script. The DSI "parameters" are the latent coordinates; the "observations" are the obs-space outputs the emulator predicts.


### Why `use_runstor=True`

There are two ways the emulator's forward run can talk to PEST++. The file-based path writes a CSV of latent coordinates, runs the emulator once, and reads a CSV of outputs — one process launch per realisation. For an emulator that is cheap per call but called thousands of times, that launch overhead dominates. The **runstor** path instead lets PESTPP-IES write all the realisations' parameters into a single run-storage file (`dsi.rns`), runs the emulator once over the whole batch, and writes every realisation's outputs back into the same file. It is invoked with the `/e` flag (`pestpp-ies dsi.pst /e`), which is what tells PEST++ to use run storage.


In [ ]:
dtd = Path("dsi_template")
dpst = dsi.prepare_pestpp(dtd, use_runstor=True)
print(f"DSI template written to {dtd}")
print(f"latent parameters: {dpst.npar}, emulated observations: {dpst.nobs}")

`prepare_pestpp` generates a single, self-contained `forward_run.py` for us (no hand-editing, no drift). Because we asked for `use_runstor=True`, its `__main__` calls `dsi_runstore_forward_run`, which loads `dsi.pickle`, reads the latent coordinates from the `.rns`, predicts the whole batch in one call, and writes the outputs back. Let us confirm the generated script is wired the way we expect:


In [ ]:
frun = (dtd / "forward_run.py").read_text()
assert "dsi_runstore_forward_run" in frun, \
    "forward_run.py is not runstor-wired; re-run prepare_pestpp(use_runstor=True)"
# show the __main__ entry point
print(frun.splitlines()[-2])
print(frun.splitlines()[-1])

### Set the targets, weights and noise

The control file comes out with the truth as obsvals and zero weights. We set the **synthetic truth** as the conditioning targets, then apply weights only to the history-period observations (day ≤ 252) of the conditioning species — nothing after the decision date informs the history match, and the held-back cations stay at zero weight (we cash those in later, in the dataworth notebook).


In [ ]:
dobs = dpst.observation_data
dobs["time"] = pd.to_numeric(dobs["time"], errors="coerce")

# the synthetic truth becomes the conditioning target
dobs.loc[truth.columns, "obsval"] = truth.values[0]
dobs["weight"] = 0.0

# conditioning species (held-back cations excluded); pe never conditioned on
cond_species = ["so4", "o0", "no3", "ph", "tmp"]  # o0 = dissolved O2 in this model
is_cond = dobs.variable.isin(cond_species)
is_history = dobs.time <= 252
nz = dobs.loc[is_cond & is_history & (dobs.obsval < 1e30)].index

Rather than re-derive a noise model here, we **lift the weights and per-observation standard deviations straight from the part1_03 control file** — the same species-specific budget (proportional-with-floor for concentrations, absolute for pH and temperature) we set when we chose the observations. part1_03 is the single source of truth for measurement noise; the DSI conditioning simply reuses it for the observations the two control files share, so the emulator is held to exactly the same data-fit standard as the full model would be.

In [ ]:
# Lift the weights and per-obs sigma straight from the part1_03 "observations,
# weights and truth" control file -- the full-order-model (FOM) pst. part1_05
# does NOT re-derive a noise model: part1_03 is the single source of truth for
# measurement noise and weights. We copy standard_deviation and weight for the
# conditioning observations the two control files share (DSI keeps only the
# stress-period-boundary times, a subset of part1_03's weighted obs). No sqrt(n)
# deflation and no emulator-error inflation -- the emulator's fidelity was
# checked above; here we trust part1_03's budget as it stands.
fom_file = Path("..", "part1_03_obs_weights_and_truth", "obs_template", "pest.pst")
if not fom_file.exists():
    raise Exception(
        "you need to run the '../part1_03_obs_weights_and_truth/"
        "dizon_obs_weights_and_truth.ipynb' notebook (its staged control file -- "
        f"the FOM weights -- was not found at '{fom_file}')")

fom_pst = pyemu.Pst(str(fom_file))
fom_pst.try_parse_name_metadata()
fom = fom_pst.observation_data
fom["weight"] = pd.to_numeric(fom["weight"], errors="coerce")
fom_w = fom.loc[fom.weight > 0]

# the conditioning obs shared by both control files
common = [o for o in nz if o in fom_w.index]
missing = [o for o in nz if o not in fom_w.index]
dobs.loc[common, "standard_deviation"] = pd.to_numeric(
    fom.loc[common, "standard_deviation"], errors="coerce").values
dobs.loc[common, "weight"] = fom.loc[common, "weight"].values
nz = pd.Index(common)   # condition only on obs that carry a part1_03 weight

print(f"lifted weights + sigma for {len(common)} conditioning obs from the part1_03 FOM pst")
if missing:
    print(f"  ({len(missing)} DSI conditioning obs had no part1_03 weight -- skipped)")

Group the observations per site:species so PEST++ can balance phi across them (no single well-species pair dominates the objective), and split the phi budget evenly across groups:


In [ ]:
# Balance phi per site:species group, exactly as part1_03 does: each group gets
# an equal share of the objective, so no single well-species pair dominates the
# fit. This per-species rebalancing is also what caps a long, autocorrelated
# series to "one group's worth" of influence -- the over-counting cure that
# makes a separate sqrt(n) weight deflation unnecessary. It is the mechanism
# part1_03 relies on, so we carry it through here rather than strip it.
dobs["obgnme"] = dobs.obsid.astype(str) + ":" + dobs.variable.astype(str)

weighted_groups = dobs.loc[nz, "obgnme"].unique()
phi_factors = pd.Series(1.0 / len(weighted_groups), index=weighted_groups)
ies_phi_factor_file = "ies_phi_factors.csv"
phi_factors.to_csv(dtd / ies_phi_factor_file, header=False)
dpst.pestpp_options["ies_phi_factor_file"] = ies_phi_factor_file
print(f"{len(weighted_groups)} site:species groups, each given phi share "
      f"{phi_factors.iloc[0]:.4f} (sum = {phi_factors.sum():.3f})")

Build the **observation noise ensemble** and hand it to PESTPP-IES. We redraw part1_03's noise model around the synthetic truth: one correlated draw per site:species series (a single instrument's error is a bias across its record, not fresh scatter at each reading), scaled by each observation's lifted standard deviation, with concentrations truncated at zero so no realisation invents a negative measurement. Passing it via `ies_observation_ensemble` is what makes the smoother condition against *exactly* this noise rather than redrawing its own:

In [ ]:
n_reals = 500
dpst.pestpp_options["ies_num_reals"] = n_reals
dpst.pestpp_options["save_binary"] = True

# Redraw the observation noise around the synthetic truth using part1_03's noise
# model: one correlated shock per site:species series per realisation (a single
# instrument's error behaves like a bias across its record, not independent
# scatter at each reading), scaled per-time by the sigma lifted from the FOM
# pst, with concentrations truncated at zero (physical bounds). This is
# part1_03's make_noise_ensemble, recentred on the truth.
ABS_SPECIES = {"ph", "tmp"}   # absolute-sigma species; everything else is a concentration
noise = pyemu.ObservationEnsemble.from_gaussian_draw(dpst, num_reals=n_reals)
nzobs = dobs.loc[nz].copy()
rng = np.random.default_rng(20260605)
for grp, g in nzobs.groupby("obgnme"):
    z = rng.standard_normal(noise.shape[0])
    vals = g.obsval.values[None, :] + z[:, None] * g.standard_deviation.values[None, :]
    if grp.split(":")[-1] not in ABS_SPECIES:      # concentrations stay >= 0
        np.clip(vals, 0.0, None, out=vals)
    noise.loc[:, g.index] = vals
noise._df = noise._df.dropna(axis=1)
noise.to_binary(str(dtd / "noise.jcb"))

# hand the ensemble straight to PESTPP-IES (otherwise the smoother draws its own)
dpst.pestpp_options["ies_observation_ensemble"] = "noise.jcb"
print(f"noise ensemble: {noise.shape[0]} reals x {noise.shape[1]} obs "
      "(redrawn around the truth, passed via ies_observation_ensemble)")

### A solver option we deliberately do *not* use

An earlier draft of this curriculum set `ies_multimodal_alpha = 0.99` here — per-realisation solves, intended (in full-model PESTPP-IES work) to protect against outlier realisations and prior-data conflict. We tested it on this emulator-space problem against two controls, and the evidence ran the other way: with the multimodal solve the posterior tightened well past what the data can justify and shifted away from the truth; without it, the posterior closely matched a nearest-neighbour reference built directly from the training ensemble. Per-realisation solves earn their keep when individual realisations can genuinely fail or conflict — neither happens inside a linear emulator. So: default solver, deliberately.

In [ ]:
# default IES solver -- no multimodal options (see the cell above for why)
dpst.pestpp_options.pop("ies_multimodal_alpha", None)
dpst.pestpp_options["ies_num_threads"] = psutil.cpu_count(logical=False)

#dpst.pestpp_options['ies_multimodal_alpha'] = 0.999
dpst.pestpp_options['ies_autoadaloc'] = True
dpst.pestpp_options['ies_subset_size'] = -100

dpst.control_data.noptmax = 1
dpst.write(str(dtd / "dsi.pst"), version=2)

### Run it (cheap — this is live)

Copy the platform binaries in and run PESTPP-IES against the emulator in run-storage mode (`/e`). Unlike every other run in this curriculum, this one is genuinely cheap: there is no DIZON in the loop, only the emulator. Three iterations over a 500-member ensemble finish in seconds to a couple of minutes, so we run it live rather than loading a prebaked result.


In [ ]:
hbd.get_bins(str(dtd))
pyemu.os_utils.run("pestpp-ies dsi.pst /e", cwd=str(dtd))

## 4. The payoff: the posterior forecast distribution (and the compute bill)

Load the conditioned ensemble. Iteration 0 is the prior (the emulator's unconditioned spread), and the last iteration is the DSI posterior — the forecast distribution after conditioning on the history-period data. We read the payoff as the **change in the forecast distribution** from prior to posterior: its median and its P95 (the quantile the operator designs treatment capacity to), and where the truth sits relative to both.


In [ ]:
rpst = pyemu.Pst(str(dtd / "dsi.pst"))
obsen = rpst.ies.obsen
# iterations available on disk (0 = prior, max = posterior)
iters = sorted(obsen.index.get_level_values(0).unique()) \
    if isinstance(obsen.index, pd.MultiIndex) else None
oe_prior = obsen.loc[0].copy()
last_it = max(i for i in [0, 1, 2, 3] if (dtd / f"dsi.{i}.obs.jcb").exists())
oe_post = obsen.loc[last_it].copy()
print(f"prior: {oe_prior.shape[0]} reals | posterior (iter {last_it}): "
      f"{oe_post.shape[0]} reals")

In [ ]:
# forecast under prior and posterior (emulated), plus the truth
prior_fc = peak_so4_mgl(oe_prior)
post_fc = peak_so4_mgl(oe_post)
truth_fc = float(peak_so4_mgl(truth).iloc[0])

# the payoff: the prior -> posterior shift in the forecast distribution
print(f"synthetic truth peak SO4: {truth_fc:.1f} mg/L")
print(f"prior     peak SO4 (mg/L): median={prior_fc.median():.0f}  "
      f"P5={prior_fc.quantile(0.05):.0f}  P95={prior_fc.quantile(0.95):.0f}")
print(f"posterior peak SO4 (mg/L): median={post_fc.median():.0f}  "
      f"P5={post_fc.quantile(0.05):.0f}  P95={post_fc.quantile(0.95):.0f}")

shift = post_fc.median() - prior_fc.median()
tighten = (post_fc.quantile(0.95) - post_fc.quantile(0.05)) - \
          (prior_fc.quantile(0.95) - prior_fc.quantile(0.05))
direction = "up" if shift > 0 else "down"
print(f"\nconditioning moved the forecast median {prior_fc.median():.0f} -> "
      f"{post_fc.median():.0f} mg/L ({direction}) and the 5-95 band "
      f"{'tightened' if tighten < 0 else 'widened'} by {abs(tighten):.0f} mg/L")
covered = post_fc.quantile(0.05) <= truth_fc <= post_fc.quantile(0.95)
print(f"truth ({truth_fc:.1f} mg/L) {'inside' if covered else 'outside'} the posterior 5-95 band")
print("one truth is one draw: a truth picked in the prior's upper tail can land in the"
      "\nposterior's tail legitimately. Whether the posterior is honestly sized is a"
      "\nCOVERAGE question -- we test it properly below.")

In [ ]:
# the payoff figure: prior vs posterior forecast distribution, with the design P95
# of each and where the truth sits. conditioning moves the mass UP and tightens it.
fig, ax = plt.subplots(figsize=(7, 3.5))
fc_max = max(prior_fc.max(), post_fc.max())
bins = np.linspace(0, fc_max * 1.05, 40)
ax.hist(prior_fc, bins=bins, color="#D3D3D3", alpha=0.8,
        label=f"prior (median {prior_fc.median():.0f}, P95 {prior_fc.quantile(0.95):.0f})")
ax.hist(post_fc, bins=bins, color="#1f77b4", alpha=0.7,
        label=f"DSI posterior (median {post_fc.median():.0f}, P95 {post_fc.quantile(0.95):.0f})")
ax.axvline(prior_fc.quantile(0.95), color="0.4", lw=1.2, ls=":")
ax.axvline(post_fc.quantile(0.95), color="#1f77b4", lw=1.2, ls=":")
ax.axvline(truth_fc, color="fuchsia", lw=1.5, label=f"synthetic truth ({truth_fc:.0f} mg/L)")
ax.set_xlabel("peak SO$_4$ at supply well (mg/L)")
ax.set_ylabel("count")
ax.set_title("payoff: prior vs DSI posterior forecast (the dotted lines are the design P95)",
             loc="left", fontsize="small")
ax.legend(fontsize="small")
fig.tight_layout()


A risk statement is a lens on this same distribution, not a different analysis. Suppose your supply contract carried a **90 mg/L trigger** — a clause that costs you above that level. Reading the exceedance probability straight off the prior and posterior forecasts:


In [ ]:
# one illustrative risk lens on the same distribution: a 90 mg/L contract trigger
TRIGGER_MGL = 90.0
p_prior = (prior_fc > TRIGGER_MGL).mean()
p_post = (post_fc > TRIGGER_MGL).mean()
print(f"P(peak SO4 > {TRIGGER_MGL:.0f} mg/L) prior     : {p_prior:.2f}")
print(f"P(peak SO4 > {TRIGGER_MGL:.0f} mg/L) posterior : {p_post:.2f}")
moved = "rose" if p_post > p_prior else ("fell" if p_post < p_prior else "did not move")
print(f"the contract risk {moved} with conditioning -- the same shift the "
      "distribution already showed, read through a threshold.")


## Is the posterior honestly sized? A calibration check

A single synthetic truth cannot tell you whether your posterior uncertainty is honest — a 5–95% band is *supposed* to miss one truth in ten. But the emulator makes the proper test nearly free: condition on several held-out realisations as if each were the truth, and count how often the posterior band covers the realisation's actual peak. Systematic misses mean the noise budget is still too confident; systematic over-coverage means it is too timid. This habit — *calibrate your posterior like you would calibrate an instrument* — is cheap insurance the full model could never afford:

In [ ]:
import shutil as _sh

CAL_TRUTHS = list(validation.index[:6])   # held-out reals the emulator never saw
cal = []
for tr in CAL_TRUTHS:
    wd = Path(f"calib_{tr}")
    if wd.exists():
        _sh.rmtree(wd)
    _sh.copytree(dtd, wd)
    # a used runstor template must be cleaned before re-running
    for patt in ("dsi.rns", "dsi.rec", "frun.out", "dsi.obs+noise.jcb"):
        (wd / patt).unlink(missing_ok=True)
    for f in list(wd.glob("dsi.[0-9].*")) + list(wd.glob("dsi.phi.*")):
        f.unlink()

    cpst = pyemu.Pst(str(wd / "dsi.pst"))
    cobs = cpst.observation_data
    # this realisation's values become the conditioning target
    cobs.loc[nz, "obsval"] = validation.loc[tr, nz].values
    # noise: same lifted budget, recentred on this truth, correlated within each
    # series, concentrations truncated at zero (species read off the group label)
    cnoise = pyemu.ObservationEnsemble.from_gaussian_draw(cpst, num_reals=n_reals)
    grp_obs = cobs.loc[nz]
    for grp, g in grp_obs.groupby("obgnme"):
        z = rng.standard_normal(cnoise.shape[0])
        vals = g.obsval.values[None, :] + z[:, None] * g.standard_deviation.values[None, :]
        if grp.split(":")[-1] not in ABS_SPECIES:
            np.clip(vals, 0.0, None, out=vals)
        cnoise.loc[:, g.index] = vals
    cnoise._df = cnoise._df.dropna(axis=1)
    cnoise.to_binary(str(wd / "noise.jcb"))
    cpst.write(str(wd / "dsi.pst"), version=2)

    pyemu.os_utils.run("pestpp-ies dsi.pst /e", cwd=str(wd))

    rp = pyemu.Pst(str(wd / "dsi.pst"))
    pen = rp.ies.obsen
    last = max(int(f.name.split(".")[1]) for f in wd.glob("dsi.[0-9].obs.jcb"))
    pk = peak_so4_mgl(pen.loc[last])
    actual = float(true_peak.loc[tr])
    lo, hi = pk.quantile(0.05), pk.quantile(0.95)
    cal.append((tr, actual, lo, hi, lo <= actual <= hi))
    print(f"truth {tr}: actual {actual:5.1f}  posterior 5-95 [{lo:5.1f}, {hi:5.1f}]"
          f"  {'covered' if cal[-1][-1] else 'missed'}")
    _sh.rmtree(wd)

n_cov = sum(c[-1] for c in cal)
print(f"\ncoverage: the posterior 5-95 band covered {n_cov} of {len(cal)} held-out "
      "truths (an honestly-sized band should cover roughly 9 in 10)")

If coverage comes back systematically low, the remedy is more humility in the noise budget (a larger emulator-error allowance, or stronger weight deflation) — *not* a different truth. The forecast distribution is only worth carrying into decisions if its stated confidence survives this kind of audit.

(The EU drinking-water standard for sulfate is 250 mg/L; the supplied water is comfortably below it under both prior and posterior, so cost, not compliance, drives this decision.)


And the contrast that is the whole point of this curriculum. Conditioning the forecast the conventional way — PESTPP-IES against DIZON, three iterations of a few hundred ~6 min runs — is **days** of compute. Conditioning the *emulator*, to the same forecast question, took the run you just watched: **seconds to minutes**, after the one-time prior MC that we had already paid for. We sharpened the forecast — moved its median and its design P95 — for almost nothing on top of the prior Monte Carlo, which was the only expensive ingredient. In the notebooks that follow, DSI lets us re-use that same investment to ask new questions (dataworth, optimization) for almost nothing more.

The lesson worth carrying forward: conditioning did not make the forecast *lower*, it made it *truer* — the prior median under-predicted the truth, and the data moved the forecast up toward it while tightening the spread. The value of data is truer news, not better news. The ["full-model check"](../part1_06_full_model_check/dizon_full_model_check.ipynb) is where we confront how trustworthy the posterior we just produced really is.


In [ ]:
ies_runs_full = 3 * post_fc.shape[0]  # rough: noptmax iters x ensemble size
min_per_run = 6
days_full = ies_runs_full * min_per_run / 60 / 24
print(f"conventional IES against DIZON: ~{ies_runs_full} runs x {min_per_run} min "
      f"= ~{days_full:.1f} days of compute")
print("DSI conditioning against the emulator: seconds to minutes (just run above)")

Next: ["full-model check"](../part1_06_full_model_check/dizon_full_model_check.ipynb) validates this DSI posterior against a genuine 201-realisation full-model history match — the expensive run we did *not* have to do here, loaded so we can see where emulator and full model agree, and where they do not.
